---

# Conference Popularity and Association Pattern Mining pada Dataset DBLP

*Notebook 04: Task 4, Frequent Pattern Mining dengan FP-Growth.*

---

## Kelompok Tugas Besar Data Mining

- Izhar Alif Akbar (18223129)
- Harfhan Ikhtiar (18223123)
- Mahesa Andre (13523140)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Implementasi FP-Growth**](#3)
4. [**Eksperimen Empat Min Support (Task 4.1)**](#4)
5. [**Validasi dengan mlxtend**](#5)
6. [**Analisis Frequent Itemsets (Task 4.2 sampai 4.5)**](#6)
7. [**Interpretasi (Task 4.6)**](#7)
8. [**Ringkasan dan Langkah Berikutnya**](#8)

---

# Introduction <a name="1"></a>

---

## Overview

*Task 4 mencari kombinasi konferensi yang sering muncul bersama dalam riwayat publikasi author. Notebook ini mengerjakan sisi FP-Growth pada author-based transactional database dari notebook `02_transactions`. Sisi Apriori dikerjakan pada notebook `03_apriori`, dan perbandingan keduanya pada notebook `06_performance`.*

## Aim

*Task 4.1: menemukan frequent itemsets pada min support 0,1%, 0,5%, 1%, dan 2%.*

*Task 4.2 dan 4.3: memisahkan 1-, 2-, dan 3-itemsets, lalu menampilkan 10 itemsets dengan support tertinggi.*

*Task 4.4 dan 4.5: menganalisis pengaruh min support terhadap jumlah itemsets dan mengidentifikasi kombinasi konferensi yang paling sering muncul bersama.*

*Task 4.6: menginterpretasikan pola yang ditemukan.*

## Metric

***Support** itemset $X$ adalah proporsi transaksi yang memuat seluruh item pada $X$. Penyebutnya selalu seluruh transaksi author-based, yaitu 1.821.757, termasuk transaksi yang dilewati saat membangun FP-tree.*

$$\text{support}(X) = \frac{|\{t \in D : X \subseteq t\}|}{|D|}$$

*Itemset frequent jika support-nya minimal min support. Dalam hitungan transaksi, ambangnya adalah bilangan bulat terkecil $c$ dengan $c / |D| \ge \text{min\_sup}$.*

## Approach

*FP-Growth memadatkan database menjadi FP-tree dalam dua pemindaian, lalu menambang pola secara rekursif dari conditional pattern base tanpa membangkitkan kandidat. Implementasi mengikuti aturan bab metodologi: item yang tidak frequent dipangkas lebih dulu, transaksi dengan kurang dari dua item frequent tidak dimasukkan ke pohon (support 1-itemset tetap dari pemindaian pertama), dan waktu pembentukan pohon dilaporkan terpisah dari waktu penambangan.*

*Kode algoritma berada di `src/fpgrowth.py` agar dapat dipakai ulang oleh notebook 05, 06, dan 07, dan sudah diuji dengan `pytest` (contoh buku teks, brute force, dan `mlxtend`). Isi modul ditampilkan pada bagian 3.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

Notebook ini tidak memakai GPU. Prasyaratnya adalah notebook `02_transactions` sudah dijalankan sehingga `data/processed/tx_author.parquet` dan `tx_items.csv` tersedia.

Sel berikut memasang semua pustaka yang dipakai notebook ini.

In [1]:
%pip install -q numpy pandas scipy matplotlib pyarrow mlxtend

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-7\data-mining\tubes\conference-pattern-mining-dblp\.venv\Scripts\python.exe -m pip install --upgrade pip


## Import Libraries

Seluruh `import` untuk notebook ini dikumpulkan di satu sel. Modul proyek di `src/` diimpor setelah `Settings` menetapkan path-nya.

In [2]:
import inspect
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

## Seed Everything

FP-Growth bersifat deterministik. Seed tetap ditetapkan agar konsisten dengan notebook lain.

In [3]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

Seluruh path, ambang min support, dan warna dipusatkan di kelas `Settings`. Palet warna sama dengan notebook sebelumnya.

In [4]:
class Settings:
    SEED = 42
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

    SRC_DIR = ROOT / "src"
    PROC_DIR = ROOT / "data" / "processed"
    TAB_DIR = ROOT / "outputs" / "tables"
    FIG_DIR = ROOT / "outputs" / "figures"
    FPM_DIR = ROOT / "outputs" / "fpgrowth"

    ITEMS = PROC_DIR / "tx_items.csv"
    TX_AUTHOR = PROC_DIR / "tx_author.parquet"

    MIN_SUPPORTS = [0.001, 0.005, 0.01, 0.02]
    TOP_K = 10

    INK = "#0b0b0b"
    INK_2 = "#52514e"
    MUTED = "#898781"
    GRID = "#e1e0d9"
    BASELINE = "#c3c2b7"
    SURFACE = "#fcfcfb"
    BLUE = "#2a78d6"
    BLUE_LIGHT = "#9ec5f4"
    ORANGE = "#eb6834"
    GRAY = "#b9b8b2"

CFG = Settings()
for d in (CFG.TAB_DIR, CFG.FIG_DIR, CFG.FPM_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CFG.SRC_DIR))
import fpgrowth as fpgrowth_module
from fpgrowth import fpgrowth, render_tree, to_frame
from fpm_data import load_horizontal, load_item_names
from validation import compare_itemsets, mlxtend_reference

Gaya grafik dan fungsi bantu diatur sekali di sini. Angka ditulis dengan pemisah ribuan titik dan desimal koma agar konsisten dengan notebook sebelumnya.

In [5]:
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "figure.facecolor": CFG.SURFACE, "axes.facecolor": CFG.SURFACE, "savefig.facecolor": CFG.SURFACE,
    "font.size": 9.5, "text.color": CFG.INK, "axes.labelcolor": CFG.INK_2,
    "xtick.color": CFG.INK_2, "ytick.color": CFG.INK_2, "axes.edgecolor": CFG.BASELINE,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": CFG.GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False,
})
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)


def fmt_id(value, digits=0):
    text = f"{value:,.{digits}f}"
    return text.replace(",", "X").replace(".", ",").replace("X", ".")


def pct_id(value, digits=1):
    return f"{value:.{digits}f}".replace(".", ",") + "%"


def save_fig(fig, name):
    fig.savefig(CFG.FIG_DIR / name)
    plt.show()


def save_table(df, name, index=True):
    df.to_csv(CFG.TAB_DIR / name, index=index)


def id_ticks(ax, axis="x"):
    formatter = FuncFormatter(lambda v, _: fmt_id(v, 0 if float(v).is_integer() else 1))
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(formatter)


def tag(ms):
    return f"{ms * 100:g}".replace(".", "p")

## Load Dataset

Tabel panjang author-based diubah ke bentuk horizontal, yaitu list frozenset kode item per transaksi. Bentuk ini sama dengan masukan Apriori pada notebook 03. Waktu pemuatan tidak termasuk pengukuran.

In [6]:
transactions = load_horizontal(CFG.TX_AUTHOR)
names = load_item_names(CFG.ITEMS)
N = len(transactions)
print(f"Transaksi: {fmt_id(N)} | item unik: {fmt_id(len(names))}")

Transaksi: 1.821.757 | item unik: 4.373


---

# Implementasi FP-Growth <a name="3"></a>

---

## 3.1 Kode

Seluruh isi `src/fpgrowth.py` ditampilkan di bawah. Setiap node FP-tree menyimpan `item`, `count`, `parent`, dan `children`, dan header table memetakan item ke daftar node-nya.

In [7]:
print(inspect.getsource(fpgrowth_module))

"""FP-Growth (Han, Pei, dan Yin, 2000) untuk transactional database berbentuk horizontal."""
from __future__ import annotations

import math
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, FrozenSet, Iterable, Iterator, List, Mapping, Optional, Sequence, Tuple

import pandas as pd


def min_count_for(min_support: float, n_transactions: int) -> int:
    """Count terkecil c sehingga c / n_transactions >= min_support (minimal 1)."""
    c = math.ceil(min_support * n_transactions)
    while c > 0 and (c - 1) / n_transactions >= min_support:
        c -= 1
    while c / n_transactions < min_support:
        c += 1
    return max(c, 1)


def order_items(items: Iterable[int], counts: Mapping[int, int]) -> Tuple[int, ...]:
    """Urutan item di FP-tree: count menurun, lalu id item menaik."""
    return tuple(sorted(items, key=lambda i: (-counts[i], i)))


class FPNode:
    __slots__ = ("item", "count", "parent", "childr

## 3.2 Contoh Kecil

Sebelum dijalankan pada data DBLP, FP-Growth diperiksa pada contoh buku teks Han, Kamber, dan Pei (2012), sembilan transaksi dengan item I1 sampai I5 dan min support 2/9. Hasil yang benar adalah 13 frequent itemsets.

In [8]:
toy = [frozenset(t) for t in ([1, 2, 5], [2, 4], [2, 3], [1, 2, 4], [1, 3], [2, 3], [1, 3], [1, 2, 3, 5], [1, 2, 3])]
toy_names = {i: f"I{i}" for i in range(1, 6)}
toy_result = fpgrowth(toy, 2 / 9)
print(render_tree(toy_result.tree, toy_names))
to_frame(toy_result, toy_names)[["names", "length", "count", "support"]]

I2:7
  I1:4
    I3:2
      I5:1
    I4:1
    I5:1
  I3:2
  I4:1
I1:2
  I3:2


,names,length,count,support
0,I2,1,7,0.777778
1,I1,1,6,0.666667
2,I3,1,6,0.666667
3,I1 + I2,2,4,0.444444
4,I1 + I3,2,4,0.444444
5,I2 + I3,2,4,0.444444
6,I4,1,2,0.222222
7,I5,1,2,0.222222
8,I1 + I5,2,2,0.222222
9,I2 + I4,2,2,0.222222


---

# Eksperimen Empat Min Support (Task 4.1) <a name="4"></a>

---

FP-Growth dijalankan sekali untuk setiap min support. Waktu pembentukan pohon (`t_build`) dan waktu penambangan (`t_mine`) dicatat terpisah. Pengukuran berulang untuk perbandingan performa dilakukan pada notebook 06. FP-tree tidak disimpan setelah penambangan (`keep_tree=False`) agar memori cukup untuk validasi.

In [9]:
results = {}
for ms in CFG.MIN_SUPPORTS:
    res = fpgrowth(transactions, ms, keep_tree=False)
    results[ms] = res
    print(f"min_sup {pct_id(ms * 100)} | min_count {fmt_id(res.min_count)} | itemsets {fmt_id(len(res.itemsets))} "
          f"| build {res.t_build:.2f} s | mine {res.t_mine:.2f} s")

frames = {ms: to_frame(res, names) for ms, res in results.items()}
for ms, df in frames.items():
    df.assign(itemset=df["itemset"].apply(list)).to_parquet(CFG.FPM_DIR / f"itemsets_{tag(ms)}.parquet", index=False)

min_sup 0,1% | min_count 1.822 | itemsets 1.514 | build 6.02 s | mine 4.35 s


min_sup 0,5% | min_count 9.109 | itemsets 88 | build 2.44 s | mine 0.59 s


min_sup 1,0% | min_count 18.218 | itemsets 30 | build 2.04 s | mine 0.13 s


min_sup 2,0% | min_count 36.436 | itemsets 6 | build 1.73 s | mine 0.00 s


---

# Validasi dengan mlxtend <a name="5"></a>

---

Setiap hasil dibandingkan dengan `mlxtend.frequent_patterns.fpgrowth` pada database yang sama. Itemset dan count harus identik.

In [10]:
checks = pd.DataFrame([
    {"min_support": ms, **compare_itemsets(res.itemsets, mlxtend_reference(transactions, ms))}
    for ms, res in results.items()
])
save_table(checks, "t4_fpgrowth_validasi_mlxtend.csv", index=False)
assert checks["identical"].all(), "hasil FP-Growth berbeda dengan mlxtend"
checks

,min_support,n_ours,n_ref,missing,extra,count_mismatch,identical
0,0.001,1514,1514,0,0,0,True
1,0.005,88,88,0,0,0,True
2,0.010,30,30,0,0,0,True
3,0.020,6,6,0,0,0,True
